# 02 - Resource attributes: deployment and identity

Goal: attach where the agent runs and what it is to **every span**, before export.

Two things about an OTel setup do not change from request to request: the deployment (environment, cluster, pod) and the agent's identity (name, id, version, framework).
OpenTelemetry has a place for exactly that: the **Resource**. Everything in the resource is attached to every span the process emits, and LangSmith shows it in the run's metadata.

You set the resource once, in `otel_setup.configure(resource_attributes=...)`. No span code changes.

Because the provider is global, `configure()` can run once per Python process. If you change your function, use **Kernel > Restart Kernel and Run All Cells**.

In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'pyproject.toml').exists())

import inspect
import json
import os
import sys
import time
import uuid
from importlib.metadata import version

for path in (str(ROOT), str(ROOT / "helpers")):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=True)

from langsmith import Client
from opentelemetry import baggage, context, trace
from opentelemetry.sdk.trace import SpanProcessor

import attr_coverage as coverage
import bank_agent
import bank_traffic
import enrich
import enrichment_ref as ref
import lab_curate
import lab_links
import otel_setup

client = Client()
PROJECT = os.environ["LANGSMITH_PROJECT"]
print("project:", PROJECT)

## Step 1: Deployment constants

In production these come from the deployment: environment variables, the Kubernetes downward API.

In [ ]:
DEPLOYMENT = ref.DEPLOYMENT
print(DEPLOYMENT)

## Step 2: Write `resource_attributes`

Return one dict: everything in `DEPLOYMENT`, plus `agent_name`, `agent_id`, `agent_version`, `agent_framework` (`"langchain"`), and `agent_framework_version`
(the installed version, from `importlib.metadata.version("langchain")`, already imported as `version`).

In [ ]:
# Reference solution: try writing this yourself first.
def resource_attributes():
    """Challenge 2: process-wide constants, attached to every span."""
    return {
        **DEPLOYMENT,
        "agent_name": "support-assistant",
        "agent_id": "bank-support-agent",
        "agent_version": "2026.10.1",
        "agent_framework": "langchain",
        "agent_framework_version": version("langchain"),
    }

In [ ]:
resource_attributes()

## Step 3: Configure OTel with it, and run the agent

The only change from challenge 1 is the argument to `configure()`.

In [ ]:
otel_setup.configure(resource_attributes=resource_attributes())
agent = bank_agent.build_agent()
bank_traffic.run_turns(bank_agent, agent, bank_traffic.QUESTIONS[:6])
otel_setup.flush()
n = bank_traffic.wait_for_runs(PROJECT, minimum=6)
print(f"{n} root runs in {PROJECT}")

## Step 4: See it on the traces

Every run in the new traces, not just the root, carries the keys. LangSmith stores resource attributes under an `otel.resource.` prefix (for example `otel.resource.environment`), and `enrich.meta_get` accepts either spelling. If `None` appears, wait a few seconds and re-run: ingestion lags.

In [ ]:
time.sleep(10)
runs = enrich.agent_runs(enrich.project_runs(client, PROJECT, hours=1))
for r in runs[:6]:
    m = enrich.meta_of(r)
    print(r.name[:28].ljust(30), enrich.meta_get(r, "environment"), enrich.meta_get(r, "agent_name"), enrich.meta_get(r, "agent_framework"), enrich.meta_get(r, "agent_framework_version"))
report = coverage.gap_report(client, PROJECT, write=True)
present, total, missing = coverage.hands_on_coverage(report)
print(f"hands-on attributes present so far: {present}/{total} (challenges 3 to 7 add the rest)")
lab_links.project_link(hint="filter on metadata.otel.resource.environment = uat, then open a run's Metadata tab")

**Result:** runs show `uat`, `support-assistant` and `langchain`. Next: challenge 3 adds values that change with every request.